# Punto de control: capítulo 5

Todo el código de la calculadora hasta este capítulo. Ejecute las celdas en orden y continúe con el capítulo siguiente.

In [ ]:
import math

In [ ]:
# Ejemplo A. Unidades: kN y m
nodos = {
    1: {"x": 0.0, "y": 0.0},
    2: {"x": 6.0, "y": 0.0},
    3: {"x": 3.0, "y": 4.0},
}

In [ ]:
# E en kN/m² (acero) y área en m²
barras = {
    1: {"nodo_inicial": 1, "nodo_final": 2,
        "elasticidad": 2.0e8, "area": 0.0020},
    2: {"nodo_inicial": 2, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
    3: {"nodo_inicial": 1, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
}

In [ ]:
# True: el desplazamiento en esa dirección está impedido
apoyos = {
    1: {"x": True, "y": True},   # articulado
    2: {"x": False, "y": True},  # rodillo
}

In [ ]:
# Cargas en kN, positivas en el sentido de los ejes
cargas = {
    3: {"x": 20.0, "y": -50.0},
}

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Ejemplo B. Unidades: kN y m
nodos_b = {
    1: {"x": 0.0, "y": 0.0},
    2: {"x": 3.0, "y": 0.0},
    3: {"x": 0.0, "y": 4.0},
    4: {"x": 3.0, "y": 4.0},
}

barras_b = {
    1: {"nodo_inicial": 1, "nodo_final": 2,
        "elasticidad": 2.0e8, "area": 0.0020},
    2: {"nodo_inicial": 2, "nodo_final": 4,
        "elasticidad": 2.0e8, "area": 0.0020},
    3: {"nodo_inicial": 3, "nodo_final": 4,
        "elasticidad": 2.0e8, "area": 0.0020},
    4: {"nodo_inicial": 1, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
    5: {"nodo_inicial": 1, "nodo_final": 4,
        "elasticidad": 2.0e8, "area": 0.0020},
    6: {"nodo_inicial": 2, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
}

apoyos_b = {
    1: {"x": True, "y": True},   # articulado
    2: {"x": False, "y": True},  # rodillo
}

cargas_b = {
    3: {"x": 30.0, "y": -40.0},
    4: {"x": 0.0, "y": -40.0},
}

In [ ]:
geometria = {}
for numero, barra in barras.items():
    inicial = nodos[barra["nodo_inicial"]]
    final = nodos[barra["nodo_final"]]
    dx = final["x"] - inicial["x"]
    dy = final["y"] - inicial["y"]
    L = math.sqrt(dx**2 + dy**2)
    geometria[numero] = {"longitud": L,
                         "c": dx / L, "s": dy / L}

In [ ]:
def calcular_geometria(nodos, barras):
    """Longitud y cosenos directores de cada barra.

    Devuelve un diccionario {número de barra:
    {"longitud": L, "c": c, "s": s}}.
    """
    geometria = {}
    for numero, barra in barras.items():
        inicial = nodos[barra["nodo_inicial"]]
        final = nodos[barra["nodo_final"]]
        dx = final["x"] - inicial["x"]
        dy = final["y"] - inicial["y"]
        L = math.sqrt(dx**2 + dy**2)
        geometria[numero] = {"longitud": L,
                             "c": dx / L, "s": dy / L}
    return geometria

In [ ]:
import numpy as np

np.set_printoptions(precision=1, suppress=True)

In [ ]:
def rigidez_local(E, A, L):
    """Matriz de rigidez de la barra en sus ejes (2x2)."""
    return E * A / L * np.array([[1.0, -1.0],
                                 [-1.0, 1.0]])


def matriz_transformacion(c, s):
    """Matriz T (2 x 4) tal que u' = T u."""
    return np.array([[c, s, 0.0, 0.0],
                     [0.0, 0.0, c, s]])


def rigidez_barra(E, A, L, c, s):
    """Matriz de rigidez de la barra en ejes globales."""
    T = matriz_transformacion(c, s)
    return T.T @ rigidez_local(E, A, L) @ T

In [ ]:
def calcular_rigideces(barras, geometria):
    """Matriz de rigidez global (4 x 4) de cada barra."""
    rigideces = {}
    for numero, barra in barras.items():
        g = geometria[numero]
        rigideces[numero] = rigidez_barra(
            barra["elasticidad"], barra["area"],
            g["longitud"], g["c"], g["s"])
    return rigideces


geometria = calcular_geometria(nodos, barras)
rigideces = calcular_rigideces(barras, geometria)